In [ ]:
from datetime import datetime
from getpass import getpass

admin_rdm_url = 'https://admin.develop.rdm.example.com/'

idp_name_1 = None
idp_username_institutional_admin = "admin_user"
idp_password_institutional_admin = "admin_password"

rdm_project_name = 'TEST-グループ管理連携機能検証-{}'.format(datetime.now().strftime('%Y%m%d'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_name_1}')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_1}')
(len(idp_username_institutional_admin), len(idp_password_institutional_admin))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# アドオン利用制御

- サブシステム名: アドオン利用制御
- ページ/アドオン: アドオン利用制御-グループ管理連携機能
- 機能分類: グループ管理連携機能
- シナリオ名: 
- 用意するテストデータ: アカウント(機関管理者1)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=False, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGakunin RDM管理者のトップページを表示する

- 管理者トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('.login-logo')).to_be_visible(timeout=30000)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

- 管理者ページが表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(page, idp_name_1, idp_username_institutional_admin, idp_password_institutional_admin, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "GakuNin RDM機関管理者ページへようこそ"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオン利用制御」を選択する

- 「アドオン利用制御」が表示されること
- 「Groups」がデフォルトで無効になっていること

In [ ]:
async def _step(page):
    # Wait for the link with the text "アドオン利用制御" to be visible
    await page.wait_for_selector('a:has-text("アドオン利用制御")')

    # Click the link
    await page.locator('a:has-text("アドオン利用制御")').click()

    # Wait for the checkbox to be present
    checkbox = page.locator('input[type="checkbox"][data-addon-short-name="groups"]')
    await expect(checkbox).to_be_visible(timeout=transition_timeout*2)

    # Scroll to the Groups item so the auto-captured screenshot shows its state
    await asyncio.sleep(1)
    await checkbox.scroll_into_view_if_needed()

    # Assert that the checkbox is not checked
    assert not await checkbox.is_checked()
    await asyncio.sleep(2)

await run_pw(_step)

## 「Groups」アドオンを有効にする

- チェックボックスがオンになること
- 「Groups」が有効になっていること

In [ ]:
async def _step(page):
    # Wait for the checkbox to be present
    checkbox = page.locator('input[type="checkbox"][data-addon-short-name="groups"]')
    await checkbox.check()

    # Scroll to the Groups item so the auto-captured screenshot shows its state
    await asyncio.sleep(1)
    await checkbox.scroll_into_view_if_needed()

    # Assert that the checkbox is checked
    assert await checkbox.is_checked()
    await asyncio.sleep(2)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}